<a href="https://colab.research.google.com/github/juanpajedrez/learn_rag_Huggingface/blob/main/Capstone_Project_Multimodal_Data_Starter_File.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment setup

In [ ]:
# Install necessary libraries
!pip install -q openai langchain langchain-openai langchain-community openai-whisper sentence-transformers pdf2image
!apt-get install poppler-utils
!pip install --upgrade Pillow

In [ ]:
%cd /content/drive/MyDrive/GenAI/RAG/CAPSTONE PROJECT - MultiModal Starbucks Finance

In [ ]:
from google.colab import userdata
api_key = userdata.get('genai_course')

In [ ]:
# Import libraries
from langchain_openai import ChatOpenAI
from openai import OpenAI
from IPython.display import display, Markdown
from sentence_transformers import SentenceTransformer
import whisper
import pandas as pd
import base64
from pdf2image import convert_from_path
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity
import os
import torch

# Audio Transcription

In [ ]:
# Check if the GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Checking which models are available
whisper.available_models()

In [ ]:
# Transcribe the audio using Whisper
model = whisper.load_model(name = 'large-v3-turbo',
                           device = device)
input_file = "starbucks-q3.mp3"
result = model.transcribe(audio = input_file)

In [ ]:
# Check the transcription output
transcription_text= result['text']
print("Transcription text:")
print(transcription_text)

In [ ]:
# Save the transcription to a text file
os.makedirs("transcript", exist_ok = True)
with open("transcript/transcript.txt", "w") as f:
  f.write(transcription_text)

# Embedding the Audio

In [ ]:
# Loading the transcription text into chunks of 100 characters
with open("transcript/transcript.txt", "r") as f:
  text = f.read()

chunk_size = 250
audio_chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

# Summary of the the chunks
print(f"Number of chunks: {len(audio_chunks)}")
print(f"First chunk: {audio_chunks[3]}")

In [ ]:
# Load the Clip model and embed the chunks
clip_model = SentenceTransformer('clip-ViT-B-32', device = device)
audio_embeddings = clip_model.encode(audio_chunks)

In [ ]:
# Check the shape of the embeddings
print(f"Shape of embeddings: {audio_embeddings.shape}")

# PDF to image transformation

In [ ]:
# Defining the paths for the PDf and the images
pdf_path = "3Q24-Earnings-Release.pdf"
output_folder = "images"
os.makedirs(output_folder, exist_ok = True)

In [ ]:
# Convert each page of PDF to images
images = convert_from_path(pdf_path)
image_paths = []

for i, image in enumerate(images):
  image_path = os.path.join(output_folder, f"page_{i+1}.jpg")
  image.save(image_path, "JPEG")
  image_paths.append(image_path)

print(f"Number of images/pages is {len(image_paths)}")
print(f"Examples of 3 image paths {image_paths[:3]}")

# Embedding the images

In [ ]:
# Using the clip model to embed images
image_embeddings = []

for filename in os.listdir(output_folder):
  if filename.endswith(".jpg"):
    image_path = os.path.join(output_folder, filename)
    image = Image.open(image_path)
    embedding = clip_model.encode(image)
    image_embeddings.append(embedding)

In [ ]:
# Check the image embeddings
print(f"Shape of image embeddings: {len(image_embeddings)}")
print(f"The shape of embeddings is {image_embeddings[0].shape}")

# Retrieval system

In [ ]:
# Defining a query
query = "what are the short term risks for the company"
query_embeddings = clip_model.encode(query)

In [ ]:
# Compute the similarity with the transcription embeddings
audio_similarities = cosine_similarity([query_embeddings], audio_embeddings)[0]

In [ ]:
# Order the top K (20) most similar audios
k = 20
top_k_audio_similarities = audio_similarities.argsort()[-k:][::-1]

In [ ]:
# Show the most similar audios and their similariries
print(f"The top 5 most similar chunks are {top_k_audio_similarities[:5]}")
print(f"The corresponding similarities are {audio_similarities[top_k_audio_similarities[:5]]}")

In [ ]:
# Comput the cosine similarity with the image embeddings
image_similarities = cosine_similarity([query_embeddings], image_embeddings)[0]

In [ ]:
# print the top k images in terms of similarities
k = 5
top_k_image_similarities = image_similarities.argsort()[-k:][::-1]

In [ ]:
# Display the top 5 images
print(f"The top 5 most similar images are {top_k_image_similarities[:5]}")
print(f"The corresponding similarities are {image_similarities[top_k_image_similarities[:5]]}")

# Prepare the context

In [ ]:
# Combine the top k transcriptions
k = 5
text_context = ' '.join([audio_chunks[idx] for idx in top_k_audio_similarities[:k]])

In [ ]:
# Retrieve the top k images and transform into base64
base64frame = []
for idx in top_k_image_similarities[:k]:
  image_path = image_paths[idx]
  with open(image_path, "rb") as image_file:
    base64frame.append(base64.b64encode(image_file.read()).decode('utf-8'))
base64frame

# Generative System

In [ ]:
# Connecting to the openAI API
client = OpenAI(api_key = api_key)

In [ ]:
# Define the system prompt
system_prompt = f"""
You are a financial advisor expert in publicly traded companies.
you must answer the {query}
You explain in clear terms with the data available only"""

In [ ]:
# PRepare the list of images
image_data_list = [{"type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{img}",
                                  "detail": "high"}} for img in base64frame]
image_data_list

In [ ]:
# Prepare the message content
user_message_content = [
    {"type": "text", "text": text_context},
    *image_data_list
]

In [ ]:
# Generate the answer
response = client.chat.completions.create(
    model = "gpt-4o-mini",
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message_content}],
    temperature = 0.2)

In [ ]:
# Display the generated answer
display(Markdown(response.choices[0].message.content))